<a href="https://colab.research.google.com/github/RatanakamonS/Stock_Price/blob/main/New_COde28Jan2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

from scipy.optimize import minimize, linprog
from scipy.stats import norm

In [ ]:
# =========================================================
# 0) CONFIG
# =========================================================
BETA = 0.95
TAIL = 1 - BETA
TOL  = 1e-6

# --- Assets (cleaned adjusted close) ---
URL_ASSETS = "https://raw.githubusercontent.com/RatanakamonS/Stock_Price/main/3yrs_clean_sp500_adjusted_close_prices.csv"

# --- Market (^GSPC) OHLCV exported to CSV on GitHub ---
URL_GSPC = "https://raw.githubusercontent.com/RatanakamonS/Stock_Price/main/gspc_ohlcv.csv"

In [ ]:
pd.read_csv(URL_ASSETS).head()

,Date,MMM,AOS,ABT,ABBV,ACN,ADBE,AMD,AES,AFL,...,WMB,WTW,WDAY,WYNN,XEL,XYL,YUM,ZBRA,ZBH,ZTS
0,1/11/2022,93.027718,51.696602,93.682190,131.623550,268.846100,316.019989,59.660000,23.080242,62.491047,...,29.147665,212.742279,151.600006,66.096024,59.473701,101.846893,111.562614,238.300003,109.038651,147.889969
1,2/11/2022,91.216164,50.844456,92.484177,129.482193,260.230713,301.220001,58.630001,22.633358,62.378803,...,28.799013,211.713242,143.509995,65.384995,58.941223,100.621338,110.826088,236.029999,104.992607,142.699524
2,3/11/2022,91.208733,51.478828,90.984261,129.392624,245.359009,285.929993,60.110001,22.948811,61.705353,...,28.894896,209.626205,140.220001,64.751892,58.796833,103.419869,112.837341,227.320007,102.779465,126.992493
3,4/11/2022,92.686195,53.003220,92.512459,130.163101,249.447037,285.750000,62.189999,23.185389,62.837128,...,29.234838,212.751938,132.630005,68.969330,59.157829,103.458473,114.990227,230.559998,102.223740,129.442505
4,7/11/2022,92.567406,53.883759,93.861435,132.689697,257.002258,299.540008,63.080002,23.465790,63.435764,...,29.496325,216.098892,136.520004,70.401123,58.887081,104.925293,115.679535,236.360001,104.836609,133.499969


In [ ]:
pd.read_csv(URL_GSPC).head()

,Price,Adj Close,Close,High,Low,Open,Volume
0,Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
1,Date,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-11-01,3856.10009765625,3856.10009765625,3911.7900390625,3843.800048828125,3901.7900390625,4481210000
3,2022-11-02,3759.68994140625,3759.68994140625,3894.43994140625,3758.679931640625,3852.89990234375,4899000000
4,2022-11-03,3719.889892578125,3719.889892578125,3750.590087890625,3698.14990234375,3733.25,4625290000


In [ ]:
# =========================================================
# INPUT FILES
# =========================================================
URL_SHARES = "https://raw.githubusercontent.com/RatanakamonS/Stock_Price/main/sp500_shares_outstanding_proxy.csv"
URL_PRICES = URL_ASSETS

# =========================================================
# 1) Load shares outstanding (proxy)
# =========================================================
shares_df = pd.read_csv(URL_SHARES)

shares_df = shares_df.rename(columns={
    "symbol": "Ticker",
    "sharesOutstanding_proxy": "Shares"
})

shares_df = shares_df.dropna(subset=["Shares"])

# =========================================================
# 2) Load prices
# =========================================================
prices = pd.read_csv(URL_PRICES, index_col=0)
prices.index = pd.to_datetime(prices.index, errors="coerce")
prices = prices.sort_index()

# ใช้วันสุดท้ายที่มีข้อมูลจริง
target_date = prices.index.max()
print("Using price date:", target_date.date())

prices_t = prices.loc[target_date].reset_index()
prices_t.columns = ["Ticker", "Price"]

# =========================================================
# 3) Merge shares × price
# =========================================================
mkt_df = shares_df.merge(prices_t, on="Ticker", how="inner")

# Market Capitalization
mkt_df["MarketCap"] = mkt_df["Shares"] * mkt_df["Price"]

# Market portfolio weights
total_mkt_cap = mkt_df["MarketCap"].sum()
mkt_df["Weight"] = mkt_df["MarketCap"] / total_mkt_cap

# =========================================================
# 4) Output
# =========================================================
mkt_df = mkt_df.sort_values("Weight", ascending=False)

print(mkt_df.head(10))
print("Sum of weights:", mkt_df["Weight"].sum())

mkt_df.to_csv("sp500_market_portfolio_weights.csv", index=False)

Using price date: 2025-12-09
    Ticker       Shares        Price     MarketCap    Weight
326   NFLX   4237323340  1188.439941  5.035804e+12  0.080559
340   NVDA  24305000000   177.820007  4.321915e+12  0.069139
310   MSFT   7432377655   509.899994  3.789769e+12  0.060626
38    AAPL  14697926000   234.070007  3.440344e+12  0.055036
22    AMZN  10690216011   228.149994  2.438973e+12  0.039017
71    AVGO   4741273799   359.254456  1.703324e+12  0.027249
304   META   2177889269   755.080383  1.644481e+12  0.026307
19   GOOGL   5818000000   240.800003  1.400974e+12  0.022412
433   TSLA   3325819167   395.940002  1.316825e+12  0.021066
20    GOOG   5407000000   241.380005  1.305142e+12  0.020879
Sum of weights: 1.0


In [ ]:
# =========================================================
# 1) Load assets (Adjusted Close) from GitHub  [WORKING]
# =========================================================
print("Loading asset prices from GitHub...")

prices = pd.read_csv(URL_ASSETS, index_col=0)

# explicit datetime parse (กัน index เป็น string)
prices.index = pd.to_datetime(prices.index, errors="coerce", dayfirst=True)
prices = prices[~prices.index.isna()].sort_index()

# ตรวจชนิด index (ต้องเป็น datetime64)
print("Index dtype:", prices.index.dtype)
print("Index sample:", prices.index[:3].tolist())

prices = prices.apply(pd.to_numeric, errors="raise")

asset_ret = prices.pct_change().dropna()
tickers = asset_ret.columns.tolist()

start = asset_ret.index.min()
end   = asset_ret.index.max()

print(f"Assets loaded: N={asset_ret.shape[1]}, T(full)={asset_ret.shape[0]}, range={start.date()} to {end.date()}")
print("asset_ret range:", asset_ret.index.min(), "->", asset_ret.index.max(), "rows:", len(asset_ret))



Loading asset prices from GitHub...
Index dtype: datetime64[ns]
Index sample: [Timestamp('2022-11-01 00:00:00'), Timestamp('2022-11-02 00:00:00'), Timestamp('2022-11-03 00:00:00')]
Assets loaded: N=495, T(full)=751, range=2022-11-02 to 2025-10-30
asset_ret range: 2022-11-02 00:00:00 -> 2025-10-30 00:00:00 rows: 751


In [ ]:
# =========================================================
# 2) Load market (^GSPC) from GitHub CSV + align dates
# =========================================================
print("Loading market index (^GSPC) from GitHub CSV...")

# ไฟล์มี 2 แถวบนที่ไม่ใช้ -> skiprows=[1,2]
gspc = pd.read_csv(URL_GSPC, skiprows=[1, 2])

# คอลัมน์แรกชื่อ "Price" แต่จริงๆ คือ Date -> rename เป็น Date
gspc = gspc.rename(columns={gspc.columns[0]: "Date"})

# set Date index
gspc["Date"] = pd.to_datetime(gspc["Date"], errors="coerce")
gspc = gspc.dropna(subset=["Date"]).set_index("Date").sort_index()

# ให้แน่ใจว่า Adj Close เป็น numeric
if "Adj Close" not in gspc.columns:
    raise KeyError("Market CSV missing column: 'Adj Close'")

gspc["Adj Close"] = pd.to_numeric(gspc["Adj Close"], errors="coerce")
gspc = gspc.dropna(subset=["Adj Close"])

# คำนวณ simple return ของตลาดจาก Adj Close
gspc["mkt_ret"] = gspc["Adj Close"].pct_change()
gspc_ret = gspc["mkt_ret"].dropna()

# align dates
common_dates = asset_ret.index.intersection(gspc_ret.index)
asset_ret = asset_ret.loc[common_dates]
gspc_ret  = gspc_ret.loc[common_dates]

T, N = asset_ret.shape
mu_market = float(gspc_ret.mean())

print(f"Aligned data: N={N}, T={T}")
print(f"mu_market (^GSPC from GitHub) = {mu_market:.10f}")

# optional save market return
out_mkt = gspc.loc[common_dates, ["Adj Close"]].copy()
out_mkt["mkt_ret"] = gspc_ret
out_mkt.dropna().to_csv("gspc_market_return.csv", index=True)
print("Saved: gspc_market_return.csv")


Loading market index (^GSPC) from GitHub CSV...
Aligned data: N=495, T=751
mu_market (^GSPC from GitHub) = 0.0008092511
Saved: gspc_market_return.csv


In [ ]:
# =========================================================
# 3) Parameters + moments
# =========================================================
mu = asset_ret.mean().values            # (N,)
Sigma = asset_ret.cov().values          # (N,N)
R = asset_ret.values                    # (T,N)

In [ ]:
# =========================================================
# 4) Common helpers
# =========================================================
def portfolio_variance(w: np.ndarray, Sigma: np.ndarray) -> float:
    w = np.asarray(w, dtype=float).reshape(-1)
    return float(w @ Sigma @ w)

def portfolio_returns(R: np.ndarray, w: np.ndarray) -> np.ndarray:
    return (R @ w).astype(float)

def normal_based_var_cvar_loss(port_ret: np.ndarray, beta: float):
    """
    Normal-based VaR/CVaR ของ LOSS = -Return
    """
    tail = 1 - beta
    mu_p = float(np.mean(port_ret))
    sd_p = float(np.std(port_ret, ddof=1))
    z = norm.ppf(beta)
    VaR_loss  = (-mu_p) + sd_p * z
    CVaR_loss = (-mu_p) + sd_p * (norm.pdf(z) / tail)
    return VaR_loss, CVaR_loss

def solver_report(name, success, message, w, mu, mu_market, long_only=False, tol=1e-6):
    print("\n" + "="*80)
    print(f"Solver Report: {name}")
    print(f"success : {success}")
    print(f"message : {message}")

    if (not success) or (w is None):
        print("="*80)
        return

    sum_w = float(np.sum(w))
    ret_w = float(mu @ w)
    min_w = float(np.min(w))

    budget_ok = abs(sum_w - 1.0) <= tol
    return_ok = abs(ret_w - float(mu_market)) <= tol
    long_ok   = (min_w + tol) >= 0.0 if long_only else True

    print("-"*80)
    print(f"sum(w)          = {sum_w:.10f} | budget_ok = {budget_ok}")
    print(f"mu @ w          = {ret_w:.10f}")
    print(f"mu_market       = {float(mu_market):.10f}")
    print(f"abs(diff)       = {abs(ret_w - float(mu_market)):.10e} | return_eq_ok = {return_ok}")
    print(f"min(w)          = {min_w:.10f} | long_only_ok = {long_ok}")
    print(f"FEASIBLE(manual)= {bool(budget_ok and return_ok and long_ok)}")
    print("="*80)

In [ ]:
# =========================================================
# 5) MODEL 1/3: Mean–Variance (SLSQP) with EQUAL return constraint
# =========================================================
def solve_mv_equal_return(mu, Sigma, mu_market, long_only: bool, N: int):
    """
    Mean–Variance:
      min  w' Σ w
      s.t. sum(w) = 1
           mu'w   = mu_market
           (optional) w_i >= 0
    """
    def mv_obj(w):
        return float(w @ Sigma @ w)

    cons = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        {"type": "eq", "fun": lambda w: (mu @ w) - float(mu_market)},
    ]

    w0 = np.ones(N) / N

    if long_only:
        bounds = [(0.0, 1.0)] * N
    else:
        bounds = [(None, None)] * N

    res = minimize(
        mv_obj, w0,
        constraints=cons,
        bounds=bounds,
        method="SLSQP",
        options={"maxiter": 20000, "ftol": 1e-12}
    )

In [ ]:
# =========================================================
# 6) MODEL 2/4: CVaR LP (Rockafellar–Uryasev) with EQUAL return constraint
#     Variables: x = [w(1..N), u(1..T), alpha]
#
#     minimize: alpha + (1/(T*tail)) * sum(u_t)
#     s.t.      u_t >= -r_t'w - alpha
#               u_t >= 0
#               sum(w) = 1
#               mu'w  = mu_market
#               (optional) w_i >= 0
# =========================================================
def build_cvar_lp_equal_return(R, mu, mu_market, T, N, tail, long_only=False):
    # objective: alpha + (1/(T*tail))*sum(u)
    c = np.zeros(N + T + 1)
    c[N:N+T] = 1.0 / (T * tail)   # u part
    c[-1] = 1.0                   # alpha

    # A_ub x <= b_ub
    # u_t >= -r_t'w - alpha  <=>  -r_t'w - u_t - alpha <= 0
    A_ub = np.zeros((T, N + T + 1))
    b_ub = np.zeros(T)

    for t in range(T):
        A_ub[t, :N]    = -R[t, :]
        A_ub[t, N + t] = -1.0
        A_ub[t, -1]    = -1.0
        b_ub[t] = 0.0

    # A_eq x = b_eq
    A_eq = np.zeros((2, N + T + 1))
    b_eq = np.zeros(2)

    # (1) sum(w)=1
    A_eq[0, :N] = 1.0
    b_eq[0] = 1.0

    # (2) mu'w = mu_market
    A_eq[1, :N] = mu
    b_eq[1] = float(mu_market)

    # bounds
    w_bounds = [(0.0, None)] * N if long_only else [(None, None)] * N
    u_bounds = [(0.0, None)] * T
    alpha_bounds = [(None, None)]
    bounds = w_bounds + u_bounds + alpha_bounds

    return c, A_ub, b_ub, A_eq, b_eq, bounds

def unpack_lp_solution(x, N, T):
    w = x[:N]
    u = x[N:N+T]
    alpha = x[-1]
    return w, u, alpha

def solve_cvar_equal_return(R, mu, mu_market, beta, long_only: bool):
    T, N = R.shape
    tail = 1 - beta
    c, A_ub, b_ub, A_eq, b_eq, bounds = build_cvar_lp_equal_return(
        R=R, mu=mu, mu_market=mu_market, T=T, N=N, tail=tail, long_only=long_only
    )
    res = linprog(
        c, A_ub=A_ub, b_ub=b_ub,
        A_eq=A_eq, b_eq=b_eq,
        bounds=bounds, method="highs"
    )
    if res.success:
        w, u, alpha = unpack_lp_solution(res.x, N, T)
        return res, w, u, alpha
    return res, None, None, None

In [ ]:
# =========================================================
# 7) Run all 4 models
# =========================================================
# --- Model 1: MV Allow Short
res_mv_allow = solve_mv_equal_return(mu, Sigma, mu_market, long_only=False, N=N)
print("\n" + "="*80)
print("Solver Report: Mean–Variance (Allow Short)")
print(f"success : {res_mv_allow.success}")
print(f"message : {res_mv_allow.message}")
print("="*80)
if not res_mv_allow.success:
    raise RuntimeError("MV (Allow Short) optimization failed. Check feasibility: mu_market may be unreachable.")

w_mv_allow = res_mv_allow.x
var_mv_allow = portfolio_variance(w_mv_allow, Sigma)
Rp_mv_allow = portfolio_returns(R, w_mv_allow)
VaR_mv_allow_loss, CVaR_mv_allow_loss = normal_based_var_cvar_loss(Rp_mv_allow, BETA)

# --- Model 3: MV Long-only
res_mv_long = solve_mv_equal_return(mu, Sigma, mu_market, long_only=True, N=N)
print("\n" + "="*80)
print("Solver Report: Mean–Variance (Long-only)")
print(f"success : {res_mv_long.success}")
print(f"message : {res_mv_long.message}")
print("="*80)
if not res_mv_long.success:
    raise RuntimeError("MV (Long-only) optimization failed. Check feasibility: mu_market may be unreachable under long-only.")

w_mv_long = res_mv_long.x
var_mv_long = portfolio_variance(w_mv_long, Sigma)
Rp_mv_long = portfolio_returns(R, w_mv_long)
VaR_mv_long_loss, CVaR_mv_long_loss = normal_based_var_cvar_loss(Rp_mv_long, BETA)

# --- Model 2: CVaR Allow Short
res_cvar_allow, w_cvar_allow, u_allow, alpha_allow = solve_cvar_equal_return(
    R=R, mu=mu, mu_market=mu_market, beta=BETA, long_only=False
)
solver_report(
    "CVaR (Allow Short)", res_cvar_allow.success, res_cvar_allow.message,
    w_cvar_allow, mu, mu_market, long_only=False, tol=TOL
)

VaR_allow_loss  = float(alpha_allow) if res_cvar_allow.success else np.nan
CVaR_allow_loss = float(res_cvar_allow.fun) if res_cvar_allow.success else np.nan
var_cvar_allow  = portfolio_variance(w_cvar_allow, Sigma) if res_cvar_allow.success else np.nan

# --- Model 4: CVaR Long-only
res_cvar_long, w_cvar_long, u_long, alpha_long = solve_cvar_equal_return(
    R=R, mu=mu, mu_market=mu_market, beta=BETA, long_only=True
)
solver_report(
    "CVaR (Long-only)", res_cvar_long.success, res_cvar_long.message,
    w_cvar_long, mu, mu_market, long_only=True, tol=TOL
)

VaR_long_loss  = float(alpha_long) if res_cvar_long.success else np.nan
CVaR_long_loss = float(res_cvar_long.fun) if res_cvar_long.success else np.nan
var_cvar_long  = portfolio_variance(w_cvar_long, Sigma) if res_cvar_long.success else np.nan


Solver Report: Mean–Variance (Allow Short)
success : True
message : Optimization terminated successfully

Solver Report: Mean–Variance (Long-only)
success : True
message : Optimization terminated successfully

Solver Report: CVaR (Allow Short)
success : True
message : Optimization terminated successfully. (HiGHS Status 7: Optimal)
--------------------------------------------------------------------------------
sum(w)          = 1.0000000000 | budget_ok = True
mu @ w          = 0.0008092511
mu_market       = 0.0008092511
abs(diff)       = 2.7321894747e-17 | return_eq_ok = True
min(w)          = -0.5290306818 | long_only_ok = True
FEASIBLE(manual)= True

Solver Report: CVaR (Long-only)
success : True
message : Optimization terminated successfully. (HiGHS Status 7: Optimal)
--------------------------------------------------------------------------------
sum(w)          = 1.0000000000 | budget_ok = True
mu @ w          = 0.0008092511
mu_market       = 0.0008092511
abs(diff)       = 1.8431

In [ ]:
# =========================================================
# 8) Summary tables
# =========================================================
summary = pd.DataFrame({
    "Model": [
        "Mean–Variance (Allow Short)",
        "CVaR (Allow Short)",
        "Mean–Variance (Long-only)",
        "CVaR (Long-only)",
    ],
    # objective ของแต่ละโมเดล (variance สำหรับ MV, CVaR_loss สำหรับ CVaR)
    "Objective": [
        var_mv_allow,
        CVaR_allow_loss,
        var_mv_long,
        CVaR_long_loss,
    ],
    # variance ของพอร์ตทุกโมเดล
    "Variance": [
        var_mv_allow,
        var_cvar_allow,
        var_mv_long,
        var_cvar_long,
    ],
    "VaR_0.95_Loss": [
        VaR_mv_allow_loss,   # normal-based (MV allow)
        VaR_allow_loss,      # alpha from LP (CVaR allow)
        VaR_mv_long_loss,    # normal-based (MV long)
        VaR_long_loss,       # alpha from LP (CVaR long)
    ],
    "CVaR_0.95_Loss": [
        CVaR_mv_allow_loss,  # normal-based (MV allow)
        CVaR_allow_loss,     # LP objective
        CVaR_mv_long_loss,   # normal-based (MV long)
        CVaR_long_loss,      # LP objective
    ],
    "Solver_success": [
        bool(res_mv_allow.success),
        bool(res_cvar_allow.success),
        bool(res_mv_long.success),
        bool(res_cvar_long.success),
    ],
    "Solver_message": [
        str(res_mv_allow.message),
        str(res_cvar_allow.message),
        str(res_mv_long.message),
        str(res_cvar_long.message),
    ]
})

weights = pd.DataFrame({
    "Ticker": tickers,
    "w_MV_allow": w_mv_allow,
    "w_CVaR_allow": (w_cvar_allow if res_cvar_allow.success else np.full(N, np.nan)),
    "w_MV_long": w_mv_long,
    "w_CVaR_long": (w_cvar_long if res_cvar_long.success else np.full(N, np.nan)),
})

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(summary.to_string(index=False))

print("\n" + "="*80)
print("WEIGHTS (first 20 rows)")
print("="*80)
print(weights.head(20).to_string(index=False))

# optional save
summary.to_csv("summary_objectives.csv", index=False)
weights.to_csv("weights_all_models.csv", index=False)

print("\nSaved: summary_objectives.csv, weights_all_models.csv")


SUMMARY
                      Model  Objective  Variance  VaR_0.95_Loss  CVaR_0.95_Loss  Solver_success                                                  Solver_message
Mean–Variance (Allow Short)   0.000007  0.000007       0.003644        0.004776            True                            Optimization terminated successfully
         CVaR (Allow Short)   0.001513  0.000021       0.001513        0.001513            True Optimization terminated successfully. (HiGHS Status 7: Optimal)
  Mean–Variance (Long-only)   0.000029  0.000029       0.008122        0.010391            True                            Optimization terminated successfully
           CVaR (Long-only)   0.009866  0.000035       0.007646        0.009866            True Optimization terminated successfully. (HiGHS Status 7: Optimal)

WEIGHTS (first 20 rows)
Ticker  w_MV_allow  w_CVaR_allow    w_MV_long  w_CVaR_long
   MMM   -0.019904      0.006406 8.379692e-20          0.0
   AOS   -0.027776      0.017904 9.399508e-20   